# QDiffCR Quantum Circuit (PQC) Visualization

This notebook defines the QDiffCR quantum bottleneck circuit and draws it.
Just run the cells top to bottom.

**Circuit:** 4 qubits, `AngleEmbedding` (RX data encoding) + `BasicEntanglerLayers` (2 layers of RX rotations + CNOT ring), read out with PauliZ. 8 trainable quantum parameters.

In [ ]:
import pennylane as qml
import numpy as np
import matplotlib.pyplot as plt

n_qubits = 4
n_layers = 2

In [ ]:
# The QDiffCR PQC
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="X")   # data encoding
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))          # variational block
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]       # readout

# example inputs (4 pooled/projected features) and the 8 trainable weights
inputs = np.array([0.5, -0.3, 0.8, -0.6])
weights = np.random.uniform(-np.pi, np.pi, (n_layers, n_qubits))
print("trainable quantum params:", weights.size)

## Text diagram (high level)

In [ ]:
print(qml.draw(circuit)(inputs, weights))

## Text diagram (expanded to gates)

In [ ]:
print(qml.draw(circuit, level="device")(inputs, weights))

## Figure: high-level circuit

In [ ]:
fig, ax = qml.draw_mpl(circuit)(inputs, weights)
fig.suptitle("QDiffCR PQC: RX AngleEmbedding + BasicEntanglerLayers (2 layers x 4 qubits)")
plt.show()

## Figure: expanded gate-level circuit

In [ ]:
fig, ax = qml.draw_mpl(circuit, level="device")(inputs, weights)
fig.suptitle("QDiffCR PQC (expanded): RX encode -> [RX + CNOT-ring] x2 -> PauliZ readout")
plt.show()

## The full QuantumBottleneckLayer (PyTorch module)

This is how the circuit is wrapped and placed at the UNet bottleneck. Uses `lightning.gpu` when running on the training machine; swap to `default.qubit` if no GPU quantum backend is available.

In [ ]:
import torch
import torch.nn as nn

class QuantumBottleneckLayer(nn.Module):
    """Global feature modulator: pool -> project to n_qubits -> PQC -> project back -> residual."""
    def __init__(self, channels, n_qubits=4, n_layers=2, backend="default.qubit"):
        super().__init__()
        self.n_qubits = n_qubits
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.pre_linear = nn.Linear(channels, n_qubits)
        self.post_linear = nn.Linear(n_qubits, channels)
        dev = qml.device(backend, wires=n_qubits)
        diff = "adjoint" if backend == "lightning.gpu" else "backprop"

        @qml.qnode(dev, interface="torch", diff_method=diff)
        def circuit(inputs, weights):
            qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="X")
            qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

        self.quantum_layer = qml.qnn.TorchLayer(circuit, {"weights": (n_layers, n_qubits)})

    def forward(self, x):
        residual = x
        z = self.pool(x).flatten(1)
        z = torch.tanh(self.pre_linear(z)) * torch.pi
        z = self.quantum_layer(z)
        z = self.post_linear(z)
        return residual + z.unsqueeze(-1).unsqueeze(-1)

# quick shape check
layer = QuantumBottleneckLayer(512)
x = torch.randn(2, 512, 16, 16)
print("output shape:", tuple(layer(x).shape))
print("total params:", sum(p.numel() for p in layer.parameters()))